<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/Nettoyage_comparaison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparaison de 4 méthodes de nettoyage

Comparaison de 4 méthodes de tokenisation/lemmatisation pour l'analyse de sentiments:
- **SpaCy Lemmes**
- **Stanza** (Stanford NLP)
- **NLTK Lemmatization**
- **TextBlob**

In [ ]:
# -*- coding: utf-8 -*-
"""
Comparaison de 4 méthodes de tokenisation/lemmatisation pour l'analyse de sentiments
- SpaCy
- Stanza (Stanford NLP)
- NLTK
- TextBlob
"""

# ============================================================================
# INSTALLATIONS ET IMPORTS
# ============================================================================

!pip install spacy pandas tqdm nltk textblob stanza matplotlib seaborn plotly wordcloud langdetect
!python -m spacy download fr_core_news_sm
!python -m textblob.download_corpora

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from tqdm import tqdm
import time
from textblob import TextBlob
import stanza
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException as LangDetectError
import warnings
warnings.filterwarnings('ignore')

# Configuration pour les graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Télécharger les ressources NLTK
print("Téléchargement des ressources NLTK...")
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

tqdm.pandas()

print("Chargement des modèles...")

# SpaCy français
nlp_spacy = spacy.load("fr_core_news_sm")
print("✓ SpaCy français chargé")

# NLTK
lemmatizer_nltk = WordNetLemmatizer()
stop_words_fr = set(stopwords.words('french'))
print("✓ NLTK configuré")

# Stanza
print("Téléchargement du modèle Stanza français...")
stanza.download('fr', verbose=False)
nlp_stanza = stanza.Pipeline('fr', verbose=False, use_gpu=False)
print("✓ Stanza français chargé")

# Mots communs français pour validation
# Mots communs français pour validation (LISTE ÉLARGIE)
MOTS_FRANCAIS_COMMUNS = {
    # Verbes fréquents
    'être', 'avoir', 'faire', 'dire', 'aller', 'voir', 'savoir', 'pouvoir',
    'falloir', 'vouloir', 'venir', 'devoir', 'prendre', 'donner', 'partir',
    'arriver', 'croire', 'demander', 'rester', 'répondre', 'porter', 'parler',
    'aimer', 'laisser', 'écouter', 'attendre', 'vivre', 'écrire', 'jouer',
    'tourner', 'regarder', 'appeler', 'continuer', 'penser', 'suivre', 'paraître',
    'sortir', 'mettre', 'lire', 'changer', 'comprendre', 'compter', 'rendre',
    'devenir', 'revenir', 'tenir', 'créer', 'ouvrir', 'commencer', 'finir',

    # Noms fréquents
    'temps', 'jour', 'homme', 'chose', 'vie', 'fois', 'monde', 'main',
    'année', 'place', 'travail', 'gouvernement', 'nombre', 'point', 'eau',
    'famille', 'enfant', 'femme', 'problème', 'service', 'état', 'groupe',
    'société', 'système', 'politique', 'économie', 'histoire', 'région',
    'développement', 'recherche', 'projet', 'marché', 'entreprise', 'niveau',
    'secteur', 'rapport', 'résultat', 'action', 'programme', 'population',
    'production', 'situation', 'question', 'information', 'formation',
    'personne', 'client', 'produit', 'activité', 'centre', 'domaine',
    'partie', 'type', 'forme', 'ordre', 'effet', 'prix', 'prix', 'conseil',

    # Adjectifs fréquents
    'grand', 'petit', 'bon', 'mauvais', 'nouveau', 'ancien', 'long', 'beau',
    'français', 'anglais', 'européen', 'national', 'international', 'social',
    'public', 'privé', 'local', 'général', 'spécial', 'important', 'possible',
    'différent', 'certain', 'vrai', 'faux', 'libre', 'entier', 'seul',
    'même', 'autre', 'tel', 'tout', 'chaque', 'plusieurs', 'premier',
    'dernier', 'prochain', 'précédent', 'suivant', 'actuel', 'futur',
    'présent', 'passé', 'disponible', 'nécessaire', 'utile', 'facile',
    'difficile', 'simple', 'complexe', 'rapide', 'lent', 'haut', 'bas',

    # Mots techniques français
    'algérie', 'algérien', 'télécommunication', 'internet', 'réseau',
    'connexion', 'abonné', 'service', 'technique', 'problème', 'solution',
    'qualité', 'performance', 'efficace', 'rapide', 'stable', 'fiable',
    'améliorer', 'développer', 'installer', 'configurer', 'maintenir',
    'réparer', 'optimiser', 'moderniser', 'upgrade', 'mise', 'jour',
    'technologie', 'innovation', 'numérique', 'digital', 'communication',
    'infrastructure', 'équipement', 'matériel', 'logiciel', 'application',
    'plateforme', 'système', 'fonctionnalité', 'interface', 'utilisateur',

    # Mots du domaine des télécoms
    'fibre', 'optique', 'adsl', 'wifi', 'mobile', 'fixe', 'débit',
    'bande', 'passante', 'latence', 'ping', 'vitesse', 'téléchargement',
    'upload', 'download', 'streaming', 'navigation', 'surf', 'surfer',
    'connecter', 'déconnecter', 'synchroniser', 'paramétrer', 'configurer'
}

print("Tous les modèles sont prêts!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 24.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.0 MB/s eta 0:00:00
   ━

In [ ]:
# ============================================================================
# MÉTRIQUES DE QUALITÉ LINGUISTIQUE
# ============================================================================

def detecter_langue(texte):
    """Détecte la langue d'un texte"""
    try:
        if len(texte.strip()) < 3:
            return 'unknown'
        return detect(texte)
    except LangDetectError:
        return 'unknown'

def calculer_ratio_francais(texte_nettoye):
    """Calcule le ratio de mots français reconnus"""
    if not texte_nettoye or len(texte_nettoye.strip()) == 0:
        return 0

    mots = texte_nettoye.split()
    if len(mots) == 0:
        return 0

    mots_francais = sum(1 for mot in mots if mot.lower() in MOTS_FRANCAIS_COMMUNS)
    return mots_francais / len(mots) * 100

def analyser_qualite_lemmatisation(texte_original, texte_nettoye):
    """Analyse la qualité de la lemmatisation"""
    if not texte_nettoye or len(texte_nettoye.strip()) == 0:
        return {
            'taux_reduction': 0,
            'ratio_francais': 0,
            'langue_detectee': 'unknown',
            'nb_mots_originaux': 0,
            'nb_mots_nettoyes': 0,
            'diversite_lexicale': 0
        }

    mots_originaux = len(texte_original.split()) if texte_original else 0
    mots_nettoyes = len(texte_nettoye.split())
    mots_uniques = len(set(texte_nettoye.split()))

    return {
        'taux_reduction': ((mots_originaux - mots_nettoyes) / max(mots_originaux, 1)) * 100,
        'ratio_francais': calculer_ratio_francais(texte_nettoye),
        'langue_detectee': detecter_langue(texte_nettoye),
        'nb_mots_originaux': mots_originaux,
        'nb_mots_nettoyes': mots_nettoyes,
        'diversite_lexicale': (mots_uniques / max(mots_nettoyes, 1)) * 100
    }

def detecter_mots_anglais_suspects(texte):
    """Détecte les mots qui semblent être de l'anglais"""
    mots_anglais_suspects = {
        'the', 'and', 'for', 'are', 'but', 'not', 'you', 'all', 'can', 'had',
        'her', 'was', 'one', 'our', 'out', 'day', 'get', 'has', 'him', 'his',
        'how', 'its', 'may', 'new', 'now', 'old', 'see', 'two', 'who', 'boy',
        'man', 'she', 'use', 'way', 'will', 'with', 'have', 'this', 'that',
        'they', 'from', 'what', 'were', 'been', 'said', 'each', 'which',
        'their', 'would', 'there', 'could', 'other', 'after', 'first',
        'well', 'water', 'long', 'little', 'very', 'work', 'know', 'good'
    }

    if not texte:
        return 0

    mots = texte.lower().split()
    if len(mots) == 0:
        return 0

    mots_suspects = sum(1 for mot in mots if mot in mots_anglais_suspects)
    return (mots_suspects / len(mots)) * 100

In [ ]:
# ============================================================================
# MÉTHODES DE NETTOYAGE
# ============================================================================

def nettoyage_spacy_lemmes(texte):
    """Méthode 1: SpaCy avec lemmatisation optimisée pour le français"""
    if pd.isna(texte) or texte == "":
        return ""

    texte = str(texte)
    doc = nlp_spacy(texte)

    lemmes = []
    for token in doc:
        if (not token.is_punct and
            not token.is_space and
            not token.is_stop and
            len(token.text) > 2 and
            token.text.isalpha() and
            token.lang_ == 'fr'):  # Vérification de la langue
            lemmes.append(token.lemma_.lower())

    return ' '.join(lemmes)

def nettoyage_stanza(texte):
    """Méthode 2: Stanza - Référence pour le français"""
    if pd.isna(texte) or texte == "":
        return ""

    texte = str(texte)

    try:
        doc = nlp_stanza(texte)

        lemmes = []
        for sentence in doc.sentences:
            for word in sentence.words:
                if (len(word.text) > 2 and
                    word.text.lower() not in stop_words_fr and
                    word.text.isalpha()):
                    lemmes.append(word.lemma.lower())

        return ' '.join(lemmes)

    except Exception as e:
        return ""

def nettoyage_nltk_lemmatization(texte):
    """Méthode 3: NLTK (problématique pour le français)"""
    if pd.isna(texte) or texte == "":
        return ""

    texte = str(texte).lower()
    tokens = word_tokenize(texte, language='french')

    tokens_filtered = []
    for token in tokens:
        if (token not in string.punctuation and
            token not in stop_words_fr and
            len(token) > 2 and
            token.isalpha()):
            tokens_filtered.append(token)

    lemmes = [lemmatizer_nltk.lemmatize(token) for token in tokens_filtered]

    return ' '.join(lemmes)

def nettoyage_textblob(texte):
    """Méthode 4: TextBlob (problématique pour le français)"""
    if pd.isna(texte) or texte == "":
        return ""

    texte = str(texte)

    try:
        blob = TextBlob(texte)

        mots = []
        for mot in blob.words:
            mot_lower = mot.lower()
            if (len(mot_lower) > 2 and
                mot_lower not in stop_words_fr and
                mot_lower.isalpha()):
                mots.append(mot.lemmatize())

        return ' '.join(mots)

    except Exception as e:
        return ""

In [ ]:
# ============================================================================
# FONCTION DE TRAITEMENT PAR BATCH
# ============================================================================

def traiter_batch_methode(textes, fonction_nettoyage, batch_size=100):
    """
    Traite les textes par batch pour optimiser les performances
    """
    resultats = []

    print(f"  Traitement par batch de {batch_size} commentaires...")

    for i in tqdm(range(0, len(textes), batch_size), desc="   Progression"):
        batch = textes[i:i+batch_size]

        # Traiter chaque commentaire du batch
        batch_resultats = []
        for texte in batch:
            batch_resultats.append(fonction_nettoyage(texte))

        resultats.extend(batch_resultats)

    return resultats

print("Fonction de traitement par batch définie")

Fonction de traitement par batch définie


In [ ]:
# ============================================================================
# ANALYSE APPROFONDIE AVEC MÉTRIQUES
# ============================================================================

def analyser_methodes_approfondies(df, colonne_commentaire, sample_size=300):
    """
    Analyse approfondie avec métriques de qualité linguistique
    """
    # Échantillonnage
    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42)
        print(f"Échantillon de {sample_size} commentaires pour l'analyse")
    else:
        df_sample = df.copy()
        print(f"Analyse de tous les {len(df)} commentaires")

    # Méthodes à comparer
    methodes = {
        'spacy_lemmes': {
            'fonction': nettoyage_spacy_lemmes,
            'description': 'SpaCy (Optimisé français)',
            'couleur': '#2E8B57'
        },
        'stanza': {
            'fonction': nettoyage_stanza,
            'description': 'Stanza (Référence)',
            'couleur': '#4169E1'
        },
        'nltk_lemmatization': {
            'fonction': nettoyage_nltk_lemmatization,
            'description': 'NLTK (Problématique)',
            'couleur': '#DC143C'
        },
        'textblob': {
            'fonction': nettoyage_textblob,
            'description': 'TextBlob (Problématique)',
            'couleur': '#FF8C00'
        }
    }

    print("\n" + "="*80)
    print("ANALYSE APPROFONDIE DES MÉTHODES DE NETTOYAGE")
    print("="*80)

    resultats_detailles = {}

    for nom_methode, config in methodes.items():
        print(f"\n Test: {config['description']}")
        start_time = time.time()

        # Appliquer la méthode
        textes_nettoyes = []
        for texte in tqdm(df_sample[colonne_commentaire], desc="Traitement"):
            textes_nettoyes.append(config['fonction'](texte))

        df_sample[f'nettoye_{nom_methode}'] = textes_nettoyes

        end_time = time.time()
        temps_execution = end_time - start_time

        # Analyser la qualité pour chaque texte
        qualites = []
        for i, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Analyse qualité"):
            qualite = analyser_qualite_lemmatisation(
                row[colonne_commentaire],
                row[f'nettoye_{nom_methode}']
            )
            qualites.append(qualite)

        # Créer un DataFrame des qualités
        df_qualite = pd.DataFrame(qualites)

        # Calculer les métriques globales
        avg_ratio_francais = df_qualite['ratio_francais'].mean()
        avg_taux_reduction = df_qualite['taux_reduction'].mean()
        avg_diversite = df_qualite['diversite_lexicale'].mean()

        # Analyser les langues détectées
        langues_detectees = df_qualite['langue_detectee'].value_counts()
        taux_francais_detecte = (langues_detectees.get('fr', 0) / len(df_sample)) * 100

        # Détecter les mots anglais suspects
        mots_anglais_suspects = []
        for texte in df_sample[f'nettoye_{nom_methode}']:
            mots_anglais_suspects.append(detecter_mots_anglais_suspects(texte))

        avg_mots_anglais = np.mean(mots_anglais_suspects)

        # Calculer le score de qualité global
        score_qualite = (
            avg_ratio_francais * 0.3 +
            taux_francais_detecte * 0.3 +
            (100 - avg_mots_anglais) * 0.2 +
            avg_diversite * 0.2
        )

        resultats_detailles[nom_methode] = {
            'description': config['description'],
            'couleur': config['couleur'],
            'temps_execution': temps_execution,
            'ratio_francais_moyen': avg_ratio_francais,
            'taux_reduction_moyen': avg_taux_reduction,
            'diversite_lexicale_moyenne': avg_diversite,
            'taux_francais_detecte': taux_francais_detecte,
            'taux_mots_anglais_suspects': avg_mots_anglais,
            'score_qualite_global': score_qualite,
            'langues_detectees': langues_detectees.to_dict(),
            'nb_textes_vides': (df_sample[f'nettoye_{nom_methode}'] == '').sum(),
            'qualites_detaillees': df_qualite
        }

        print(f" Temps d'exécution: {temps_execution:.2f}s")
        print(f" Ratio mots français: {avg_ratio_francais:.1f}%")
        print(f" Langue française détectée: {taux_francais_detecte:.1f}%")
        print(f" Mots anglais suspects: {avg_mots_anglais:.1f}%")
        print(f" Score qualité global: {score_qualite:.1f}/100")

    return df_sample, resultats_detailles

In [ ]:
# ============================================================================
# VISUALISATIONS AVANCÉES
# ============================================================================

def creer_visualisations_completes(resultats_detailles, df_sample):
    """Crée des visualisations complètes pour l'analyse"""

    # Préparation des données pour les graphiques
    methodes = list(resultats_detailles.keys())
    descriptions = [resultats_detailles[m]['description'] for m in methodes]
    couleurs = [resultats_detailles[m]['couleur'] for m in methodes]


    # 2. Graphique en barres des scores de qualité
    fig_scores = go.Figure(data=[
        go.Bar(
            x=descriptions,
            y=[resultats_detailles[m]['score_qualite_global'] for m in methodes],
            marker_color=couleurs,
            text=[f"{resultats_detailles[m]['score_qualite_global']:.1f}" for m in methodes],
            textposition='auto',
        )
    ])

    fig_scores.update_layout(
        title="Score de Qualité Global (0-100)",
        xaxis_title="Méthodes",
        yaxis_title="Score de Qualité",
        yaxis=dict(range=[0, 100]),
        height=500
    )

    fig_scores.show()

    # 3. Analyse détaillée des problèmes linguistiques
    fig_problemes = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Ratio Mots Français (%)', 'Mots Anglais Suspects (%)',
                       'Temps d\'Exécution (s)', 'Textes Vides'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )

    for i, methode in enumerate(methodes):
        r = resultats_detailles[methode]

        # Ratio mots français
        fig_problemes.add_trace(
            go.Bar(x=[r['description']], y=[r['ratio_francais_moyen']],
                   name=r['description'], marker_color=r['couleur'], showlegend=False),
            row=1, col=1
        )

        # Mots anglais suspects
        fig_problemes.add_trace(
            go.Bar(x=[r['description']], y=[r['taux_mots_anglais_suspects']],
                   name=r['description'], marker_color=r['couleur'], showlegend=False),
            row=1, col=2
        )

        # Temps d'exécution
        fig_problemes.add_trace(
            go.Bar(x=[r['description']], y=[r['temps_execution']],
                   name=r['description'], marker_color=r['couleur'], showlegend=False),
            row=2, col=1
        )

        # Textes vides
        fig_problemes.add_trace(
            go.Bar(x=[r['description']], y=[r['nb_textes_vides']],
                   name=r['description'], marker_color=r['couleur'], showlegend=False),
            row=2, col=2
        )

    fig_problemes.update_layout(height=800, title_text="Analyse Détaillée des Performances")
    fig_problemes.show()

    print("Toutes les visualisations ont été générées!")


In [ ]:
# ============================================================================
# RAPPORT STATISTIQUE DÉTAILLÉ
# ============================================================================

def analyser_exemples_concrets(df_sample, resultats_detailles, nb_exemples=10):
    """Analyse des exemples concrets de nettoyage pour mieux comprendre les résultats"""

    print(f"\n ANALYSE D'EXEMPLES CONCRETS DE NETTOYAGE")
    print("=" * 70)

    # Prendre quelques exemples variés
    indices_exemples = np.random.choice(len(df_sample), min(nb_exemples, len(df_sample)), replace=False)

    for i, idx in enumerate(indices_exemples[:5], 1):  # Montrer seulement 5 exemples
        row = df_sample.iloc[idx]

        print(f"\n EXEMPLE {i}:")
        print(f"Original: '{row['commentaire'][:100]}{'...' if len(row['commentaire']) > 100 else ''}'")
        print("-" * 50)

        for methode, resultats in resultats_detailles.items():
            texte_nettoye = row[f'nettoye_{methode}']
            ratio_fr = calculer_ratio_francais(texte_nettoye) if texte_nettoye else 0

            print(f"{resultats['description']:<25}: '{texte_nettoye[:60]}{'...' if len(texte_nettoye) > 60 else ''}'")
            print(f"{'':25}  → {len(texte_nettoye.split()) if texte_nettoye else 0} mots, {ratio_fr:.1f}% français")


In [ ]:
# ============================================================================
# FONCTION PRINCIPALE D'EXÉCUTION
# ============================================================================

def executer_analyse_complete():
    """Fonction principale pour exécuter l'analyse complète"""

    # Chargement des données
    from google.colab import files
    import io

    print(" CHARGEMENT DES DONNÉES")
    print("=" * 40)
    print("Veuillez sélectionner votre fichier CSV:")

    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    print(f" Fichier '{filename}' uploadé avec succès!")

    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f" Données chargées: {len(df)} lignes, {len(df.columns)} colonnes")

    print(f"\n APERÇU DES DONNÉES:")
    print(df.head())
    print(f"\n Colonnes disponibles: {list(df.columns)}")

    # Identifier la colonne des commentaires
    colonne_commentaire = 'commentaire'  # Ajustez si nécessaire

    if colonne_commentaire not in df.columns:
        print("  ATTENTION: Colonne 'commentaire' non trouvée")
        print("Colonnes disponibles:", list(df.columns))
        # Vous pouvez ajuster ici ou demander à l'utilisateur
        return None, None, None

    # Lancer l'analyse approfondie
    df_sample, resultats_detailles = analyser_methodes_approfondies(df, colonne_commentaire, sample_size=300)

    # Créer les visualisations
    creer_visualisations_completes(resultats_detailles, df_sample)

    # Sauvegarder les résultats
    print(f"\n SAUVEGARDE DES RÉSULTATS")
    print("=" * 40)

    # Données détaillées
    df_sample.to_csv('analyse_complete_nettoyage.csv', index=False, encoding='utf-8')
    print(" Données détaillées: analyse_complete_nettoyage.csv")

    # Rapport statistique
    rapport_df = pd.DataFrame([
        {
            'Methode': resultats['description'],
            'Score_Qualite': resultats['score_qualite_global'],
            'Ratio_Francais': resultats['ratio_francais_moyen'],
            'Langue_FR_Detectee': resultats['taux_francais_detecte'],
            'Mots_Anglais_Suspects': resultats['taux_mots_anglais_suspects'],
            'Temps_Execution': resultats['temps_execution'],
            'Diversite_Lexicale': resultats['diversite_lexicale_moyenne']
        }
        for methode, resultats in resultats_detailles.items()
    ]).sort_values('Score_Qualite', ascending=False)

    rapport_df.to_csv('rapport_comparaison_methodes.csv', index=False, encoding='utf-8')
    print(" Rapport comparatif: rapport_comparaison_methodes.csv")

    # Télécharger les fichiers
    files.download('analyse_complete_nettoyage.csv')
    files.download('rapport_comparaison_methodes.csv')

    print(" Analyse terminée! Fichiers téléchargés.")

    return df_sample, resultats_detailles, classement

In [ ]:
# ============================================================================
# EXÉCUTION PRINCIPALE AVEC TOUTES LES ANALYSES
# ============================================================================

def lancer_analyse_complete_avancee():
    """Lance l'analyse complète avec toutes les visualisations et statistiques"""

    print(" LANCEMENT DE L'ANALYSE COMPLÈTE AVANCÉE")
    print("=" * 60)

    # Étape 1: Analyse principale
    df_sample, resultats_detailles, classement = executer_analyse_complete()

    if df_sample is None:
        print(" Erreur lors du chargement des données")
        return

    # Étape 2: Analyses supplémentaires
    print("\n ANALYSES SUPPLÉMENTAIRES EN COURS...")

    # D'abord, analyser des exemples concrets
    analyser_exemples_concrets(df_sample, resultats_detailles, nb_exemples=10)

    print(f"\nANALYSE TERMINÉE!")
    print("Tous les fichiers CSV ont été téléchargés")
    print("Toutes les visualisations ont été générées")
    print("Le rapport détaillé est affiché ci-dessus")

    return df_sample, resultats_detailles, classement

In [ ]:
# ============================================================================
# LANCEMENT AUTOMATIQUE
# ============================================================================

lancer_analyse_complete_avancee()

 LANCEMENT DE L'ANALYSE COMPLÈTE AVANCÉE
 CHARGEMENT DES DONNÉES
Veuillez sélectionner votre fichier CSV:


Saving resultats_analyse (1).csv to resultats_analyse (1) (2).csv
 Fichier 'resultats_analyse (1) (2).csv' uploadé avec succès!
 Données chargées: 9238 lignes, 3 colonnes

 APERÇU DES DONNÉES:
                                         commentaire sentiment  \
0                       Plantez et vous récolterez.,   positif   
1  Bravo et merci à tous les employés d’Algérie T...   positif   
2  Votre entreprise ne pense plus qu’aux abonnés ...   positif   
3  Ce qui nous réjouit, c’est le passage de Idoom...   positif   
4  On demande juste à être raccordés à la nouvell...   négatif   

          probleme  
0              NaN  
1              NaN  
2              NaN  
3              NaN  
4  problème réseau  

 Colonnes disponibles: ['commentaire', 'sentiment', 'probleme']
Échantillon de 300 commentaires pour l'analyse

ANALYSE APPROFONDIE DES MÉTHODES DE NETTOYAGE

 Test: SpaCy (Optimisé français)


Analyse qualité: 100%|██████████| 300/300 [00:01<00:00, 152.00it/s]


 Temps d'exécution: 2.65s
 Ratio mots français: 33.8%
 Langue française détectée: 74.0%
 Mots anglais suspects: 0.1%
 Score qualité global: 71.5/100

 Test: Stanza (Référence)


Analyse qualité: 100%|██████████| 300/300 [00:02<00:00, 126.46it/s]


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Analyse terminée! Fichiers téléchargés.


NameError: name 'classement' is not defined